# kin_03 — UMAP embedding of tongue movement bouts

Does UMAP reveal discrete clusters or continuous structure in movement kinematics?

**Pipeline:**
1. Load `all_tongue_movements`
2. Quality filter, feature selection
3. StandardScaler normalization
4. UMAP fit (requires `umap-learn`)
5. Colored scatter plots by kinematic features
6. Feature–UMAP axis correlation
7. Single-session overlay
8. Save embedding

## 1. Setup

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from plotstyle import apply_style, PALETTE, style_ax, save_fig
apply_style()

In [ ]:
if Path("/root/capsule").exists():
    ENV       = "codeocean"
    SCRATCH   = Path("/root/capsule/scratch")
    FOR_LOCAL = SCRATCH / "for_local"
else:
    ENV       = "local"
    FOR_LOCAL = Path("/Users/mib/Documents/Code/kinematics_analysis/data/for_local")
    SCRATCH   = FOR_LOCAL.parent

FIG_DIR  = SCRATCH / "figures" / "kin_03_umap"
SAVE_FIG = False
print(f"ENV={ENV}")

## 2. Load data

In [ ]:
if ENV == "codeocean":
    movements_path = SCRATCH / "all_tongue_movements_04022026" / "all_tongue_movements_04022026.parquet"
else:
    movements_path = FOR_LOCAL / "all_tongue_movements_04022026.parquet"

all_tongue_movements = pd.read_parquet(movements_path)
print("Shape:", all_tongue_movements.shape)
print("Sessions:", all_tongue_movements["session"].nunique())

## 3. Feature selection and quality filter

In [ ]:
UMAP_FEATURES = [
    "duration", "peak_velocity", "mean_velocity", "total_distance",
    "excursion_angle_deg", "endpoint_x", "endpoint_y",
    "out_duration", "out_peak_velocity", "out_mean_velocity", "out_total_distance",
]
# keep available features only
UMAP_FEATURES = [f for f in UMAP_FEATURES if f in all_tongue_movements.columns]

df = (
    all_tongue_movements
    .dropna(subset=UMAP_FEATURES)
    .copy()
    .reset_index(drop=True)
)
print(f"UMAP dataset: {len(df):,} movements, {len(UMAP_FEATURES)} features")
print("Features:", UMAP_FEATURES)

## 4. Normalization

In [ ]:
from sklearn.preprocessing import StandardScaler

X_raw = df[UMAP_FEATURES].to_numpy(dtype=float)
scaler = StandardScaler()
X_norm = scaler.fit_transform(X_raw)
print("X_norm shape:", X_norm.shape, "  mean≈0, std≈1:", X_norm.mean(axis=0).round(2))

## 5. UMAP fit

Requires `umap-learn`: `pip install umap-learn`.
All analysis cells degrade gracefully if not installed.

In [ ]:
try:
    import umap as umap_lib
    HAS_UMAP = True
except ImportError:
    print("umap-learn not installed. Run: pip install umap-learn")
    HAS_UMAP = False

if HAS_UMAP:
    reducer = umap_lib.UMAP(n_neighbors=15, min_dist=0.1, n_components=2,
                             random_state=42, low_memory=True, verbose=False)
    embedding = reducer.fit_transform(X_norm)
    df_embedded = df.copy()
    df_embedded["umap_0"] = embedding[:, 0]
    df_embedded["umap_1"] = embedding[:, 1]
    print(f"UMAP complete — embedding shape {embedding.shape}")
else:
    embedding = None
    df_embedded = df.copy()

## 6. Colored scatter plots

In [ ]:
def plot_umap_colored(df_emb, color_col, cmap="coolwarm", s=1.5, alpha=0.4, figsize=(7, 6)):
    """Scatter UMAP embedding colored by a DataFrame column."""
    if "umap_0" not in df_emb.columns:
        print(f"No UMAP embedding — skipping {color_col}")
        return None
    vals = df_emb[color_col].to_numpy(dtype=float) if color_col in df_emb.columns else None
    if vals is None or not np.isfinite(vals).any():
        print(f"No valid values for {color_col}")
        return None
    lo, hi = np.nanpercentile(vals[np.isfinite(vals)], [2, 98])
    vals_c  = np.clip(vals, lo, hi)

    fig, ax = plt.subplots(figsize=figsize)
    sc = ax.scatter(df_emb["umap_0"], df_emb["umap_1"], c=vals_c,
                    cmap=cmap, s=s, alpha=alpha, linewidths=0, rasterized=True)
    plt.colorbar(sc, ax=ax, label=color_col)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.set_title(color_col)
    style_ax(ax)
    plt.tight_layout()
    return fig

if HAS_UMAP:
    for col in ["out_peak_velocity", "excursion_angle_deg", "out_duration",
                "out_total_distance", "endpoint_y"]:
        if col in df_embedded.columns:
            fig = plot_umap_colored(df_embedded, col)
            if fig:
                save_fig(fig, f"umap_{col}", fig_dir=FIG_DIR, save=SAVE_FIG)
                plt.show()

## 7. Feature–UMAP axis correlation

In [ ]:
if HAS_UMAP and embedding is not None:
    from scipy.stats import spearmanr

    rows = []
    for feat in UMAP_FEATURES:
        vals = df_embedded[feat].to_numpy(dtype=float)
        for axis_i, axis_name in [(0, "UMAP_1"), (1, "UMAP_2")]:
            emb_axis = embedding[:, axis_i]
            mask = np.isfinite(vals) & np.isfinite(emb_axis)
            rho, p = spearmanr(vals[mask], emb_axis[mask]) if mask.sum() >= 5 else (np.nan, np.nan)
            rows.append({"feature": feat, "axis": axis_name, "rho": rho, "p": p})
    corr_df = pd.DataFrame(rows)
    print(corr_df.pivot(index="feature", columns="axis", values="rho").round(3).to_string())

## 8. Single-session overlay

In [ ]:
# Single-session embedding: same UMAP fitted on all sessions, colored by feature
if HAS_UMAP and embedding is not None:
    sessions = df_embedded["session"].unique()
    if len(sessions) > 0:
        sess_pick = sessions[0]
        sess_mask = df_embedded["session"] == sess_pick
        print(f"Session: {sess_pick}  n={sess_mask.sum()}")

        fig, ax = plt.subplots(figsize=(7, 6))
        ax.scatter(
            df_embedded.loc[~sess_mask, "umap_0"],
            df_embedded.loc[~sess_mask, "umap_1"],
            s=1, alpha=0.2, color="#d0d0d0", rasterized=True, label="other sessions",
        )
        ax.scatter(
            df_embedded.loc[sess_mask, "umap_0"],
            df_embedded.loc[sess_mask, "umap_1"],
            s=3, alpha=0.7, color=PALETTE["accent"], rasterized=True, label=sess_pick[:30],
        )
        ax.set_xlabel("UMAP 1")
        ax.set_ylabel("UMAP 2")
        ax.set_title("Single session highlighted in global embedding")
        ax.legend(fontsize=7)
        style_ax(ax)
        plt.tight_layout()
        save_fig(fig, "umap_single_session", fig_dir=FIG_DIR, save=SAVE_FIG)
        plt.show()

## 9. Save embedding

In [ ]:
if HAS_UMAP and embedding is not None:
    save_dir = FIG_DIR.parent / "umap"
    save_dir.mkdir(parents=True, exist_ok=True)
    embed_path = save_dir / "umap_embedding.npy"
    np.save(embed_path, embedding)
    df_embedded[["umap_0","umap_1","session"]].to_parquet(
        save_dir / "umap_coords.parquet", index=False
    )
    print(f"Saved: {embed_path}")
else:
    print("No embedding to save.")